In [ ]:
#
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import time

colors = ["gold", "mediumturquoise", "darkorange", "lightgreen"]
kaggle_colors = ["#39c5ff", "#ffffff", "#f2f2f2", "#9ca3a4", "#3f484b"]
font = "Rubik"

In [ ]:
### cell 0 ###

df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/madhurpant/beautiful-kaggle-2022-analysis/input/kaggle-survey-2022/kaggle_survey_2022_responses.csv"
)
factor = 10
df = pd.concat([df] * factor, ignore_index=True)
df.info()

In [ ]:
### cell 1 ###

df = df[1:]

In [ ]:
### cell 2 ###

df["Duration (in seconds)"] = pd.to_numeric(
    df["Duration (in seconds)"], errors="coerce"
).astype("int")

In [ ]:
### cell 3 ###

df["Q3"] = df["Q3"].replace(
    ["Nonbinary", "Prefer not to say", "Prefer to self-describe"], "Others"
)

In [ ]:
### cell 4 ###

gender_count = (
    df["Q3"].value_counts().reset_index().rename(columns={"index": "Q3", "Q3": "count"})
)

In [ ]:
### cell 5 ###

duration_count = (
    df.groupby(["Duration (in seconds)"])
    .size()
    .reset_index()
    .rename(columns={0: "count"})
)
duration_count = duration_count[:1000]

In [ ]:
### cell 6 ###

age_count = (
    df["Q2"].value_counts(sort=False).rename_axis("Q2").reset_index(name="count")
)

In [ ]:
### cell 7 ###

country_count = df.groupby(["Q4"]).size().reset_index().rename(columns={0: "count"})
country_count = country_count.sort_values(by=["count"], ascending=False).reset_index(
    drop=True
)
country_count = country_count[:10]

In [ ]:
### cell 8 ###

country_df = (
    df["Q4"].value_counts().reset_index(name="count").rename(columns={"index": "Q4"})
)

In [ ]:
### cell 9 ###

student_count = (
    df["Q5"].value_counts().reset_index().rename(columns={"index": "Q5", "Q5": "count"})
)

In [ ]:
### cell 10 ###

# Optimized for cudf: use value_counts for GPU-accelerated aggregation
degree_count = df["Q8"].value_counts().reset_index()
# rename columns without a CPU-intensive .rename()
degree_count.columns = ["Q8", "count"]
# replace the long string on the GPU rather than boolean-indexing on the CPU
degree_count["Q8"] = degree_count["Q8"].replace(
    {
        "Some college/university study without earning a bachelor’s degree": "College Without Bachelors"
    }
)

In [ ]:
### cell 11 ###

# Compute counts of each Q16 category using a single vectorized call
experience_count = (
    df["Q16"].value_counts().reset_index(name="count").rename(columns={"index": "Q16"})
)

In [ ]:
### cell 12 ###

annual_income_df = df.groupby(["Q29"]).size().reset_index().rename(columns={0: "count"})

In [ ]:
### cell 13 ###

gender_duration_count = (
    df.groupby(["Duration (in seconds)", "Q3"])
    .size()
    .reset_index()
    .rename(columns={0: "count"})
)
gender_duration_count = gender_duration_count[:3000]

In [ ]:
### cell 14 ###

country_gender = (
    df.groupby(["Q4", "Q3"]).size().reset_index().rename(columns={0: "count"})
)

In [ ]:
### cell 15 ###

age_duration_df = (
    df[df["Duration (in seconds)"] < 2000][["Q2", "Duration (in seconds)"]]
    .groupby("Q2", as_index=False)
    .mean()
    .round(1)
)

In [ ]:
### cell 16 ###

education_gender = (
    df.groupby(["Q8", "Q3"]).size().reset_index().rename(columns={0: "count"})
)

In [ ]:
### cell 17 ###

## Not Including India bcz it acts as an outlier here
top_9_countries = [
    "United States of America",
    "Other",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Mexico",
]

top_9_countries_df = df[df["Q4"].isin(top_9_countries)]

In [ ]:
### cell 18 ###

gender_student = (
    df.groupby(["Q3", "Q5"]).size().reset_index().rename(columns={0: "count"})
)